# 07 - Business Report

Цель: собрать понятный отчёт для тимлида.


In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

DATA_SYNTHETIC = ROOT / "data" / "synthetic"
DATA_PROCESSED = ROOT / "data" / "processed"
REPORTS = ROOT / "reports"
FIGURES = REPORTS / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

print("Project root:", ROOT)


Project root: /Users/andrey/Documents/projects/COMPASS-AI


## Problem

Тимлиду сложно вручную выбирать исполнителя задачи: нужно учитывать
навыки, загрузку, срочность, качество и развитие сотрудников.


## Solution

COMPASS AI анализирует задачу и команду, ранжирует кандидатов через
TaskEmployeeMatchingNet и объясняет рекомендацию на русском языке.


In [2]:
employees = pd.read_csv(DATA_SYNTHETIC / "employees.csv")
tasks = pd.read_csv(DATA_SYNTHETIC / "tasks.csv")
assignments = pd.read_csv(DATA_SYNTHETIC / "assignments.csv")

summary = {
    "employees": len(employees),
    "tasks": len(tasks),
    "assignments": len(assignments),
    "success_rate": float(assignments["success_label"].mean()),
    "average_workload": float(employees["current_workload"].mean()),
}

display(summary)


{'employees': 36,
 'tasks': 2500,
 'assignments': 60000,
 'success_rate': 0.45061666666666667,
 'average_workload': 0.6282777777777777}

In [3]:
with open(REPORTS / "model_metrics.json", encoding="utf-8") as file:
    model_metrics = json.load(file)

with open(REPORTS / "ranking_metrics.json", encoding="utf-8") as file:
    ranking_metrics = json.load(file)

display(pd.DataFrame([model_metrics]))
display(pd.DataFrame(ranking_metrics).T)


,accuracy,precision,recall,f1,roc_auc,pr_auc,positive_rate,predicted_positive_rate,rows
0,0.78104,0.775764,0.733595,0.75409,0.859216,0.820082,0.457642,0.432766,8924.0


,precision_at_1,precision_at_3,ndcg_at_3,mrr,tasks_evaluated,rows_evaluated
ml_model,0.893333,0.853333,0.862555,0.943778,375.0,8924.0
rule_based_baseline,0.872000,0.859556,0.861110,0.932667,375.0,8924.0
random_baseline,0.485333,0.501333,0.495777,0.667997,375.0,8924.0


In [4]:
from src.agents.orchestrator import recommend_synthetic_task

example = recommend_synthetic_task(
    "TASK-0001",
    mode="balanced_workload",
    top_k=3,
    use_llm=False,
)

print("Task:", example["title"])
print("Mode:", example["mode"])
print("Recommended:", example["top_candidates"][0]["name"])
print("Explanation:")
print(example["explanation"])


Task: Добавить endpoint для статистики команды — интеграции с Plane
Mode: balanced_workload
Recommended: Никита Егоров
Explanation:
## COMPASS AI Recommendation

Рекомендованный исполнитель: Никита Егоров (backend_developer).
Режим рекомендации: balanced_workload.
Score: 0.956596.

Почему top-1 подходит:
- модель оценила совпадение задачи и доступного списка кандидатов;
- учтены skill match, риск, загрузка, скорость и надёжность;
- итоговый ranking сформирован до LLM-объяснения.

Альтернативные кандидаты:
- 2. Полина Васильева (backend_developer, senior) — score=0.956551, skill=1.0, risk=0.0668, workload_pressure=0.525
- 3. Сергей Павлов (backend_developer, middle) — score=0.925464, skill=1.0, risk=0.0668, workload_pressure=0.275

Важно: это recommendation support. Финальное решение остаётся за тимлидом.


In [5]:
import plotly.express as px

workload_df = (
    employees[
        [
            "name",
            "role",
            "grade",
            "current_workload",
            "active_tasks_count",
        ]
    ]
    .sort_values("current_workload", ascending=False)
    .head(15)
)

fig = px.bar(
    workload_df,
    x="name",
    y="current_workload",
    color="role",
)
fig.update_layout(title="Team workload")
fig.show()
fig.write_html(FIGURES / "business_team_workload.html")


In [6]:
fairness_path = REPORTS / "fairness_summary.json"

if fairness_path.exists():
    with open(fairness_path, encoding="utf-8") as file:
        fairness_summary = json.load(file)

    display(fairness_summary)
else:
    print("Run 05_fairness_analysis.ipynb first")


{'recommended_tasks': 375,
 'senior_recommendation_share': 0.9066666666666666,
 'junior_recommendation_share': 0.0,
 'top_employee_concentration': 0.4826666666666667,
 'average_recommended_workload': 0.4831226666666667}

## Conclusion

COMPASS AI показывает не только accuracy, но и управленческую зрелость:
учитывает риски, загрузку, развитие и качество прошлых назначений.
